In [1]:
from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive/StoryGenerator"):
    drive.mount('/content/drive')
    os.chdir("drive/My Drive/StoryGenerator")

KeyboardInterrupt: 

In [ ]:
!pip install -U BitsandBytes
!pip install -U transformers accelerate peft sentence-transformers sentencepiece
!pip install triton
!pip install numpy

!pip install transformers torch lm-format-enforcer huggingface_hub optimum
!pip install auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu118/ 

In [2]:
import torch, bitsandbytes as bnb
print("Torch version:", torch.__version__)
print("BitsandBytes version:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.9.0+cu126
BitsandBytes version: 0.48.2
CUDA available: True
GPU: Tesla T4


In [ ]:
from transformers import (
    BitsAndBytesConfig,
    TextIteratorStreamer,
)
import torch

MISTRAL_7B_INSTRUCT = "mistralai/Mistral-7B-Instruct-v0.3"
LLAMA_3_8B = "meta-llama/Llama-3.1-8B"
VICUNA_13B_V1_5 = "lmsys/vicuna-13b-v1.5"
QWEN_2_5_7B_INSTRUCT = "Qwen/Qwen2.5-7B-Instruct-1M"
LONG_WRITER_8B = "zai-org/LongWriter-llama3.1-8b"

quant_4_bit_config = BitsAndBytesConfig(
	load_in_4bit=True,
	bnb_4bit_use_double_quant=True,
	bnb_4bit_quant_type="nf4",
	bnb_4bit_compute_dtype=torch.float16
)

quant_8_bit_config = BitsAndBytesConfig(
	load_in_8bit=True,
	#llm_int8_skip_modules=["lm_head"]
)

In [ ]:
from llm.transformers_llm import TransformersLLM
import json
hf_token = ""
with open("cred.json", "r") as f:
	cred = json.load(f)
	hf_token = cred["hugging_face_token"]
llm = TransformersLLM(model_id=QWEN_2_5_7B_INSTRUCT, quantization_config=quant_8_bit_config, login_token=hf_token)

In [ ]:
from prompts import builder
from llm.llm import GenerationParams
from schemas import rough_outline

system_instruction, prompt = builder.build_prompt("rough_outline",
    {
		"detective": "an aging former police detective, now an occasional private investigator with sarcastic wit",
		"tropes": "murder mystery, closed room mystery, two points of view",
		"constraints": "no supernatural elements, no random hidden rooms",
		"setting": "2020s, a small rural town, an old mansion on a hill",
		"tone": "tense and suspenseful, but humorous and witty"
	}
)

generation_params = GenerationParams(
    max_tokens=10000,
    temperature=0.9,
    top_p=0.9,
    top_k=20,
    response_type="application/json",
    response_schema=rough_outline.RoughOutline
)

ro_response = llm.generate_stream(prompt, system_instruction, generation_params)
rough_outline = rough_outline.RoughOutline.model_validate_json(ro_response.text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/872 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

Model loaded in 8-bit quantized mode


In [ ]:
from serialization import StoryDirectory
from utils import information_extraction

title = information_extraction.extract_title(rough_outline)
print(title)
story_directory = StoryDirectory.new(title)
story_directory.save_stage(
    stage="rough_outline",
    prompt=prompt,
    response=ro_response,
    system_instruction=system_instruction,
    model=llm.model_id,
    generation_params=generation_params
)

In [ ]:
system_instruction, prompt = builder.build_prompt("detailed_outline", 
	{
		"rough_outline": rough_outline,
	}
)

generation_params = GenerationParams(
	max_tokens=15000,
	temperature=1,
	top_p=0.9,
	top_k=20,
)

do_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
detailed_outline = do_response.text

In [ ]:
story_directory.save_stage(
	stage="detailed_outline",
	prompt=prompt,
	response=do_response,
	system_instruction=system_instruction,
	model=llm.model_id,
	generation_params=generation_params
)

In [ ]:
import time

chapter_outlines = [ch for ch in (c.strip() for c in detailed_outline.split("\n\n")) if ch and len(ch) > 100]

chapters = []
for i, chapter_outline in enumerate(chapter_outlines):
    system_instruction, prompt = builder.build_prompt("chapter",
		{
			"rough_outline": rough_outline,
			"previous_chapter": chapter_outlines[i-1] if i > 0 else "N/A",
			"next_chapter": chapter_outlines[i+1] if i < len(chapter_outlines) - 1 else "N/A",
			"current_chapter": chapter_outline,
			"index": i + 1
		}
	)
    generation_params = GenerationParams(
		temperature=1,
		top_p=0.9,
		top_k=20,
	)
    print(f"\n\n--- Generating Chapter {i+1} ---\n\n")
    ch_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
    chapter = ch_response.text
    chapters.append(chapter)
    story_directory.save_stage(
		stage=f"chapter_{i+1:02d}",
		prompt=prompt,
		response=ch_response,
		system_instruction=system_instruction,
		model=llm.model_id,
		generation_params=generation_params
	)

In [ ]:
story_directory.save_plain_text(
	stage="full_story",
	text="\n\n".join(f"{index}: {name}\n{chapter}" for index, name, chapter in zip([f"Chapter {i+1}" for i in range(len(chapters))], [chapter_summary.title for chapter_summary in detailed_outline.chapters], chapters))
)